In [1]:
# 1. Import thư viện xử lý dữ liệu
import pandas as pd
import numpy as np

# 2. Import công cụ xử lý đường dẫn
from pathlib import Path

# 3. Import mô hình Decision Tree
from sklearn.tree import DecisionTreeClassifier

# 4. Import các công cụ đánh giá mô hình
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 5. Import công cụ lưu mô hình
import joblib

# 6. Xác định thư mục gốc project
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "processed" / "train.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

# 7. Xác định thư mục dữ liệu
DATA_DIR = PROJECT_DIR / "data" / "processed"

# 8. Đọc dữ liệu đã chia tập
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# 9. Kiểm tra kích thước
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (315125, 5)
Validation: (67524, 5)
Test: (67524, 5)


In [2]:
# 1. Danh sách 10 đặc trưng sử dụng cho mô hình
FEATURE_COLUMNS = [
    "url_length",
    "hostname_length",
    "dot_count",
    "hyphen_count",
    "digit_count",
    "at_count",
    "slash_count",
    "question_count",
    "equal_count",
    "has_https"
]

# 2. Tên cột nhãn cần dự đoán
TARGET_COLUMN = "type"


# 3. Hàm trích xuất đặc trưng từ URL
def extract_features(data):

    features = pd.DataFrame(index=data.index)

    # Độ dài URL
    features["url_length"] = data["normalized_url"].str.len()

    # Độ dài hostname
    features["hostname_length"] = data["hostname"].str.len()

    # Số dấu chấm
    features["dot_count"] = data["normalized_url"].str.count(r"\.")

    # Số dấu gạch ngang
    features["hyphen_count"] = data["normalized_url"].str.count("-")

    # Số chữ số
    features["digit_count"] = data["normalized_url"].str.count(r"\d")

    # Số ký tự @
    features["at_count"] = data["normalized_url"].str.count("@")

    # Số dấu /
    features["slash_count"] = data["normalized_url"].str.count("/")

    # Số dấu ?
    features["question_count"] = data["normalized_url"].str.count(r"\?")

    # Số dấu =
    features["equal_count"] = data["normalized_url"].str.count("=")

    # URL có sử dụng HTTPS hay không
    features["has_https"] = (
        data["normalized_url"]
        .str.lower()
        .str.startswith("https://")
        .astype(int)
    )

    return features

In [3]:
# 1. Trích xuất đặc trưng từ tập Train
train_features = extract_features(train_df)

# 2. Trích xuất đặc trưng từ tập Validation
val_features = extract_features(val_df)

# 3. Trích xuất đặc trưng từ tập Test
test_features = extract_features(test_df)

# 4. Tách dữ liệu đầu vào X và nhãn cần dự đoán y
X_train = train_features[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_val = val_features[FEATURE_COLUMNS]
y_val = val_df[TARGET_COLUMN]

X_test = test_features[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

# 5. Kiểm tra kích thước dữ liệu
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (315125, 10)
y_train: (315125,)
X_val: (67524, 10)
y_val: (67524,)
X_test: (67524, 10)
y_test: (67524,)


In [4]:
# 1. Khởi tạo mô hình Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42
)

# 2. Huấn luyện mô hình bằng tập Train
dt_model.fit(X_train, y_train)

# 3. Thông báo kết quả
print("Huấn luyện Decision Tree hoàn tất!")

# 4. Kiểm tra số lượng nút và độ sâu của cây
print("Độ sâu của cây:", dt_model.get_depth())
print("Số lượng nút:", dt_model.tree_.node_count)

Huấn luyện Decision Tree hoàn tất!
Độ sâu của cây: 38
Số lượng nút: 8915


In [5]:
# 1. Dự đoán nhãn trên tập Validation
y_val_pred = dt_model.predict(X_val)

# 2. So sánh một số nhãn thực tế và nhãn dự đoán
comparison_df = pd.DataFrame({
    "Nhãn thực tế": y_val.values,
    "Nhãn dự đoán": y_val_pred
})

# 3. Hiển thị 10 kết quả đầu tiên
display(comparison_df.head(10))

# 4. Đếm số lượng dự đoán đúng và sai
correct = (y_val.values == y_val_pred).sum()
incorrect = (y_val.values != y_val_pred).sum()

print("Số dự đoán đúng:", correct)
print("Số dự đoán sai:", incorrect)
print("Tổng số mẫu:", len(y_val))

,Nhãn thực tế,Nhãn dự đoán
0,legitimate,legitimate
1,legitimate,legitimate
2,legitimate,legitimate
3,legitimate,legitimate
4,legitimate,legitimate
5,legitimate,legitimate
6,legitimate,legitimate
7,legitimate,legitimate
8,legitimate,legitimate
9,legitimate,legitimate


Số dự đoán đúng: 66538
Số dự đoán sai: 986
Tổng số mẫu: 67524


In [6]:
# 1. Tính Accuracy
val_accuracy = accuracy_score(y_val, y_val_pred)

# 2. Tính Precision cho lớp phishing
val_precision = precision_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 3. Tính Recall cho lớp phishing
val_recall = recall_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 4. Tính F1-score cho lớp phishing
val_f1 = f1_score(
    y_val,
    y_val_pred,
    pos_label="phishing",
    zero_division=0
)

# 5. Hiển thị kết quả
print("===== KẾT QUẢ ĐÁNH GIÁ DECISION TREE =====")
print(f"Accuracy : {val_accuracy:.4f}")
print(f"Precision: {val_precision:.4f}")
print(f"Recall   : {val_recall:.4f}")
print(f"F1-score : {val_f1:.4f}")

===== KẾT QUẢ ĐÁNH GIÁ DECISION TREE =====
Accuracy : 0.9854
Precision: 0.9636
Recall   : 0.9739
F1-score : 0.9687


In [7]:
# 1. Tạo ma trận nhầm lẫn
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=["legitimate", "phishing"]
)

# 2. Hiển thị ma trận
print("===== CONFUSION MATRIX =====")
print(cm)

# 3. Tách các giá trị trong ma trận
tn, fp, fn, tp = cm.ravel()

print("\n===== CHI TIẾT =====")
print("True Negative (TN):", tn)
print("False Positive (FP):", fp)
print("False Negative (FN):", fn)
print("True Positive (TP):", tp)

===== CONFUSION MATRIX =====
[[51284   577]
 [  409 15254]]

===== CHI TIẾT =====
True Negative (TN): 51284
False Positive (FP): 577
False Negative (FN): 409
True Positive (TP): 15254


In [8]:
# 1. Dự đoán trên tập Train
y_train_pred = dt_model.predict(X_train)

# 2. Tính Accuracy trên Train
train_accuracy = accuracy_score(y_train, y_train_pred)

# 3. Tính Precision cho lớp phishing
train_precision = precision_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 4. Tính Recall cho lớp phishing
train_recall = recall_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 5. Tính F1-score cho lớp phishing
train_f1 = f1_score(
    y_train,
    y_train_pred,
    pos_label="phishing",
    zero_division=0
)

# 6. Tạo bảng so sánh
comparison_metrics = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Train": [
        train_accuracy,
        train_precision,
        train_recall,
        train_f1
    ],
    "Validation": [
        val_accuracy,
        val_precision,
        val_recall,
        val_f1
    ]
})

# 7. Hiển thị kết quả
display(comparison_metrics.round(4))

,Chỉ số,Train,Validation
0,Accuracy,0.9975,0.9854
1,Precision,0.9997,0.9636
2,Recall,0.9897,0.9739
3,F1-score,0.9947,0.9687


In [9]:
# 1. Tạo báo cáo phân loại
report = classification_report(
    y_val,
    y_val_pred,
    labels=["legitimate", "phishing"],
    target_names=["legitimate", "phishing"],
    zero_division=0
)

# 2. Hiển thị báo cáo
print("===== CLASSIFICATION REPORT =====")
print(report)

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

  legitimate       0.99      0.99      0.99     51861
    phishing       0.96      0.97      0.97     15663

    accuracy                           0.99     67524
   macro avg       0.98      0.98      0.98     67524
weighted avg       0.99      0.99      0.99     67524



In [10]:
# 1. Xác định thư mục lưu mô hình
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

# 2. Tạo thư mục nếu chưa tồn tại
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Xác định đường dẫn lưu mô hình
MODEL_PATH = ARTIFACT_DIR / "decision_tree_baseline.joblib"

# 4. Lưu mô hình
joblib.dump(dt_model, MODEL_PATH)

# 5. Kiểm tra kết quả
print("Đã lưu mô hình thành công!")
print("Đường dẫn:", MODEL_PATH)
print("Kích thước file:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

Đã lưu mô hình thành công!
Đường dẫn: d:\HocTap\Machine_learning\phishing-detection\artifacts\decision_tree_baseline.joblib
Kích thước file: 698.26 KB
